In [1]:
pip install langchain-community langchain-ollama ragas datasets pandas sentence-transformers faiss-cpu pypdf tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 3.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 10.5 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 21.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.1/35.1 MB 33.5 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 47.4 MB/s  0:00:00
  Attempting uninstall: rich0m╺━━━━━━━━━━━━━━━━━━━━━  6/13 [scikit-network]
    Found existing installation: rich 15.0.0━━━━━━━━━━━━━━━━━━  6/13 [scikit-network]
    Uninstalling rich-15.0.0:0m╺━━━━━━━━━━━━━━━━━━━━━  6/13 [scikit-network]
      Successfully uninstalled rich-15.0.0━━━━━━━━━━━━━━━━━━━━  6/13 [scikit-network]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [ragas]m12/13 [ragas]ts]r]s]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import json
import csv
import pandas as pd
from tqdm import tqdm
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness
from langchain_ollama import ChatOllama
from langchain_community.embeddings import HuggingFaceEmbeddings

# ============================================================
# 1. SETUP LOCAL MODELS
# ============================================================
print("Loading local Ollama model and embeddings...")

local_llm = ChatOllama(
    model="qwen2.5:7b", 
    temperature=0 
)

local_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# ============================================================
# 2. CHECK EXISTING PROGRESS (CHECKPOINTING)
# ============================================================
input_file = "experiment_dataset.jsonl"
output_csv = "ragas_hallucination_results.csv"

processed_ids = set()

# If the output file already exists, read the IDs that are already done
if os.path.exists(output_csv):
    print(f"🔄 Found existing output file '{output_csv}'. Loading progress...")
    try:
        # Read the CSV to find which IDs are already processed
        df_existing = pd.read_csv(output_csv)
        if 'original_id' in df_existing.columns:
            processed_ids = set(df_existing['original_id'].astype(str))
            print(f"⏮️ Found {len(processed_ids)} items already evaluated. Skipping them...\n")
    except Exception as e:
        print(f"⚠️ Could not read existing CSV ({e}).")

# ============================================================
# 3. PREPARE TO STREAM EVALUATIONS
# ============================================================
# Count total valid lines for the progress bar
total_lines = 0
with open(input_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip(): total_lines += 1

print(f"🚀 Starting Resumable Ragas Evaluation...")

# Open the CSV in append mode ('a')
file_exists = os.path.exists(output_csv)
with open(output_csv, 'a', newline='', encoding='utf-8') as csvfile:
    # Define the CSV columns
    fieldnames = ['original_id', 'question', 'rag_answer', 'faithfulness']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    
    # Write the header row only if creating a brand new file
    if not file_exists or os.path.getsize(output_csv) == 0:
        writer.writeheader()

    with open(input_file, 'r', encoding='utf-8') as f:
        progress_bar = tqdm(f, total=total_lines, desc="🕵️ Evaluating Answers", unit="item")
        
        for line_num, line in enumerate(progress_bar, 1):
            line = line.strip()
            if not line:
                continue
            
            try:
                data = json.loads(line)
                item_id = str(data.get("id"))
                
                # ------------------------------------------------
                # CHECKPOINT TRIGGER: Skip if completed previously
                # ------------------------------------------------
                if item_id in processed_ids:
                    progress_bar.set_postfix_str(f"Skipping ID: {item_id[:6]}... Done")
                    continue
                
                rag_answer = data.get("rag_answer", "")
                question = data.get("question", "")
                
                # Skip items that failed generation
                if not rag_answer:
                    progress_bar.set_postfix_str(f"ID: {item_id[:6]}... No RAG Answer")
                    continue
                
                # Update UI
                progress_bar.set_postfix_str(f"Evaluating ID: {item_id[:6]}...")

                # Normalize evidence (The ArrowInvalid Fix)
                raw_evidence = data.get("evidence", "")
                context_list = [str(item) for item in raw_evidence] if isinstance(raw_evidence, list) else [str(raw_evidence)]

                # ============================================================
                # EVALUATE EXACTLY ONE ROW
                # ============================================================
                data_samples = {
                    "question": [str(question)],
                    "contexts": [context_list],
                    "answer": [str(rag_answer)]
                }
                
                # Create a mini dataset of 1 item
                single_item_dataset = Dataset.from_dict(data_samples)
                
                # Run Ragas on the single item
                # Note: We suppress Ragas's internal print statements so it doesn't mess up my tqdm progress bar
                result = evaluate(
                    dataset=single_item_dataset,
                    metrics=[faithfulness],
                    llm=local_llm,
                    embeddings=local_embeddings,
                    show_progress=False # Turns off Ragas' internal progress bar
                )
                
                # Extract the score (converts to pandas to safely pull the specific row's value)
                score_df = result.to_pandas()
                faithfulness_score = score_df['faithfulness'].iloc[0]

                # ============================================================
                # STREAM & FLUSH: Save to disk instantly
                # ============================================================
                writer.writerow({
                    'original_id': item_id,
                    'question': question,
                    'rag_answer': rag_answer,
                    'faithfulness': faithfulness_score
                })
                
                # Force OS to save to disk immediately
                csvfile.flush()

            except json.JSONDecodeError:
                print(f"\n⚠️ Line {line_num}: Invalid JSON format. Skipping...")
            except Exception as e:
                print(f"\n❌ Error evaluating item {data.get('id', 'Unknown')}: {e}. Skipping...")

print(f"\n🎉 Evaluation Complete! All results are securely saved in '{output_csv}'.")

/var/folders/04/ycg4c8hs6tncxvm8l19zwkj80000gn/T/ipykernel_70201/1782682263.py:8: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness


Loading local Ollama model and embeddings...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3934.76it/s]


🚀 Starting Resumable Ragas Evaluation...


🕵️ Evaluating Answers: 100%|██████████| 519/519 [4:42:58<00:00, 32.71s/item, Evaluating ID: c534ea...]  


🎉 Evaluation Complete! All results are securely saved in 'ragas_hallucination_results.csv'.


In [5]:
# ragas_hallucination_results.csv to ragas_hallucination_results.jsonl

def convert_csv_to_jsonl(csv_file, jsonl_file):
    """
    Convert a CSV file to JSONL format.
    
    Args:
        csv_file (str): Path to the input CSV file.
        jsonl_file (str): Path to the output JSONL file.
    """
    with open(csv_file, 'r', encoding='utf-8') as csvfile, open(jsonl_file, 'w', encoding='utf-8') as jsonlfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            json.dump(row, jsonlfile)
            jsonlfile.write('\n')

convert_csv_to_jsonl("ragas_hallucination_results.csv", "ragas_hallucination_results.jsonl")